## Model

In [1]:
import numpy as np

class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        # var_smoothing: fraction of the largest variance added to every
        # feature variance. Prevents log(0) on near-constant features
        # (e.g. always-black border pixels) and acts as a regulariser.
        self.var_smoothing = var_smoothing
        self.means      = None   # shape: (n_classes, n_features)
        self.variances  = None   # shape: (n_classes, n_features)
        self.log_priors = None   # shape: (n_classes,)
        self.classes    = None   # array of unique class labels

    def fit(self, X, y, class_weights=None):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # --- Log Priors ---
        # P(class) = count(class) / total, scaled by optional weight.
        # We store log-probabilities to avoid underflow when multiplying
        # many small likelihoods together later.
        self.log_priors = np.zeros(n_classes)
        for i, c in enumerate(self.classes):
            prior = np.sum(y == c) / n_samples
            if class_weights is not None:
                prior *= class_weights[c]        # up-weight minority classes
            self.log_priors[i] = np.log(prior)

        # --- Per-class means and variances ---
        # Gaussian NB assumes features are conditionally independent given
        # the class, so we only need the per-feature mean and variance.
        self.means     = np.zeros((n_classes, n_features))
        self.variances = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes):
            X_c = X[y == c]                      # samples belonging to class c

            if class_weights is not None:
                # Weighted statistics: each sample gets the weight of its class,
                # then we normalise so the weights sum to 1.
                w = np.full(len(X_c), class_weights[c])
                w = w / w.sum()
                self.means[i]     = (w[:, None] * X_c).sum(axis=0)
                self.variances[i] = (w[:, None] * (X_c - self.means[i]) ** 2).sum(axis=0)
            else:
                self.means[i]     = X_c.mean(axis=0)
                self.variances[i] = X_c.var(axis=0)

        # Smooth variances by a fraction of the largest feature variance to
        # avoid log(0) on near-constant features. Scaling by max(var) keeps
        # the smoothing proportional to the data's natural spread.
        epsilon = self.var_smoothing * np.max(self.variances)
        self.variances += epsilon

    def _log_likelihood(self, X, class_idx):
        # Gaussian log-PDF: log P(x | class) = sum over features of
        #   -0.5 * log(2π * σ²)  -  (x - μ)² / (2σ²)
        # Summing the log-PDF across features exploits the conditional
        # independence assumption (products become sums in log-space).
        mean = self.means[class_idx]
        var  = self.variances[class_idx]

        log_norm  = -0.5 * np.log(2 * np.pi * var)           # normalisation term
        log_gauss = -0.5 * ((X - mean) ** 2) / var           # exponent term
        return (log_norm + log_gauss).sum(axis=1)             # shape: (n_samples,)

    def predict(self, X):
        # Score each class: log P(class | x) ∝ log P(class) + log P(x | class)
        # We pick the class with the highest (unnormalised) log-posterior.
        log_posteriors = np.array([
            self.log_priors[i] + self._log_likelihood(X, i)
            for i in range(len(self.classes))
        ])  # shape: (n_classes, n_samples)

        best_class_indices = np.argmax(log_posteriors, axis=0)
        return self.classes[best_class_indices]

## Cross Validation

In [2]:
from preprocessing2 import preprocess, custom_macro_f1_score, k_fold_indices
import numpy as np

# Feature methods we sweep over — each one produces a different
# representation of the digit images for Gaussian NB to learn from.
feature_methods = ["cnn", "hog", "pca", "flatten","hog_pca"]

# var_smoothing is the only Gaussian NB hyperparameter. We sweep across
# 12 orders of magnitude so the search captures both the under-smoothed
# (numerical instability) and over-smoothed (priors dominate) regimes.
param_grid = {
    'var_smoothing': [1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
}

best_f1_score = 0
best_params = {}

total_runs = len(feature_methods) * len(param_grid['var_smoothing'])
current_run = 1

for feature_method in feature_methods:
    # Re-extract features for each method. preprocess() returns the
    # already-split train/val/test tensors plus class weights (unused here).
    X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess(
        feature_method=feature_method,
        n_pca=50
    )

    # 3-fold split of the training set. Each fold trains on 2/3 and
    # validates on the held-out third — gives a more stable F1 estimate
    # than a single train/val split.
    folds = k_fold_indices(X_train, k=3)

    for vs in param_grid['var_smoothing']:
        print(f"--- Run {current_run}/{total_runs} | Feature:{feature_method} | var_smoothing:{vs} ---")

        fold_f1_scores = []

        for train_idx, val_idx in folds:
            X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
            X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]

            # Fresh model per fold so statistics aren't leaked between folds.
            cv_model = GaussianNaiveBayes(var_smoothing=vs)
            cv_model.fit(X_fold_train, y_fold_train)

            preds = cv_model.predict(X_fold_val)
            # Macro-F1 weights every class equally — robust to imbalance
            # and the right metric for 10-class digit classification.
            fold_f1 = custom_macro_f1_score(y_fold_val, preds, n_classes=10)
            fold_f1_scores.append(fold_f1)

        avg_f1 = np.mean(fold_f1_scores)
        print(f"    -> 3-Fold Average Macro F1: {avg_f1:.4f}\n")

        # Track the (feature_method, var_smoothing) combo with the best
        # mean fold score — that's what we'll retrain on all training data.
        if avg_f1 > best_f1_score:
            best_f1_score = avg_f1
            best_params = {
                'feature_method': feature_method,
                'var_smoothing': vs
            }

        current_run += 1

print("="*50)
print("  GRID SEARCH COMPLETE")
print("="*50)
print(f"Best CV Macro F1: {best_f1_score:.4f}")
print(f"Best Parameters: {best_params}")

Extracting CNN Features...
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step
--- Run 1/60 | Feature:cnn | var_smoothing:1e-12 ---
    -> 3-Fold Average Macro F1: 0.5117

--- Run 2/60 | Feature:cnn | var_smoothing:1e-11 ---
    -> 3-Fold Average Macro F1: 0.5225

--- Run 3/60 | Feature:cnn | var_smoothing:1e-10 ---
    -> 3-Fold Average Macro F1: 0.5335

--- Run 4/60 | Feature:cnn | var_smoothing:1e-09 ---
    -> 3-Fold Average Macro F1: 0.5456

--- Run 5/60 | Feature:cnn | var_smoothing:1e-08 ---
    -> 3-Fold Average Macro F1: 0.5609

--- Run 6/60 | Feature:cnn | var_smoothing:1e-07 ---
    -> 3-Fold Average Macro F1: 0.5821

--- Run 7/60 | Feature:cnn | var_smoothing:1e-06 ---
    -> 3-Fold Average Macro F1: 0.6092

--- Run 8/60 | Feature:cnn | var_smoothing:1e-05 ---
    -> 3-Fold Average Macro F1: 0.6438

--- Run 9/60 | Feature:cnn | var_smoothing:0.0001 ---
    -> 3-Fold Average Macro F1: 0.6818

--- Run 10

## Testing

In [ ]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix

# Re-run preprocessing with the winning feature method so the final model
# trains on the same representation that scored best in cross-validation.
print(f"Re-extracting features with winning method: {best_params['feature_method']}")
X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess(
    feature_method=best_params['feature_method'],
    n_pca=50
)

# Train on the FULL training set (no folds now) using the winning hyperparameter.
print("Training FINAL Phase 2 model with winning CV parameters...")
final_nb = GaussianNaiveBayes(var_smoothing=best_params['var_smoothing'])
final_nb.fit(X_train, y_train)

# Evaluate on the held-out test set — first time these samples are seen.
print("\nRunning final prediction on unseen TEST data...")
final_test_preds = final_nb.predict(X_test)

print("\n" + "="*50)
print(f"  OFFICIAL PHASE 2 TEST PERFORMANCE ({best_params['feature_method'].upper()})")
print("="*50)

target_names = [f"Digit {i}" for i in range(10)]
print(custom_classification_report(y_test, final_test_preds, target_names=target_names))

print("\nOfficial Test Confusion Matrix (10x10):")
print(custom_confusion_matrix(y_test, final_test_preds, n_classes=10))

Re-extracting features with winning method: hog
Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Training FINAL Phase 2 model with winning CV parameters...

Running final prediction on unseen TEST data...

  OFFICIAL PHASE 2 TEST PERFORMANCE (HOG)
                 precision     recall   f1-score    support

Digit 0               0.92       0.96       0.94        980
Digit 1               0.92       0.97       0.95       1135
Digit 2               0.94       0.90       0.92       1032
Digit 3               0.94       0.92       0.93       1010
Digit 4               0.97       0.92       0.94        982
Digit 5               0.94       0.89       0.91        892
Digit 6               0.97       0.96       0.96        958
Digit 7               0.93       0.87       0.90       1028
Digit 8               0.82       0.90       0.86        974
Digit 9               0.87       0.91       0.89       1009

accuracy                                    0.92      10000
mac

## Bias–Variance Analysis

To understand whether the model is underfitting (high bias) or overfitting (high variance), we plot a **learning curve**: the training and validation accuracy as a function of how much training data the model sees.

### The Idea Behind Bias and Variance

Every supervised model's prediction error can be decomposed into two components:

- **Bias** measures how far the model's average prediction is from the true value. A high-bias model is too simple to capture the patterns in the data — it **underfits**. Symptoms: both training and validation accuracy are low.

- **Variance** measures how much the model's predictions change when trained on different subsets of the data. A high-variance model is too sensitive to the specific training examples — it **overfits**, memorising noise instead of learning generalisable patterns. Symptoms: training accuracy is high but validation accuracy is much lower — a visible **gap** between the two curves.

### How Learning Curves Reveal Bias and Variance

The learning curve trains the model on increasing fractions of the data (10%, 20%, ..., 100%) and evaluates on a fixed validation set:

- **High bias (underfitting):** Both curves plateau at a low accuracy and converge close together. Adding more data doesn't help because the model is too simple.
- **High variance (overfitting):** The training curve stays high while the validation curve is noticeably lower. The gap between them is the signature of overfitting.
- **Good fit:** Both curves converge at a high accuracy with a small gap between them.

### What to Expect From Gaussian Naive Bayes

Gaussian NB is inherently a **low-variance, potentially high-bias** model — it makes a strong conditional independence assumption (features are independent given the class) and fits only per-class means and variances. This means it trains instantly and generalises stably, but it may underfit when features are actually correlated. We therefore expect the training and validation curves to converge quickly and stay close together, which would confirm that the model's error is dominated by bias (the approximation) rather than variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from preprocessing2 import custom_accuracy_score

# ─────────────────────────────────────────
# LEARNING CURVE
# ─────────────────────────────────────────
train_fractions = np.linspace(0.1, 1.0, 10)
train_scores    = []
val_scores      = []

total_train_samples = len(X_train)

print("Generating Learning Curves for GaussianNaiveBayes...")
for fraction in train_fractions:
    subset_size = int(total_train_samples * fraction)
    print(f"  Training on {subset_size} samples ({int(fraction*100)}%)...")

    X_subset = X_train[:subset_size]
    y_subset = y_train[:subset_size]

    model = GaussianNaiveBayes(var_smoothing=best_params['var_smoothing'])
    model.fit(X_subset, y_subset)

    train_preds = model.predict(X_subset)
    val_preds   = model.predict(X_val)

    train_scores.append(custom_accuracy_score(y_subset, train_preds))
    val_scores.append(custom_accuracy_score(y_val,   val_preds))

# ─────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot(train_fractions * 100, train_scores,
         label="Training Accuracy",   color="blue",   marker="o", linewidth=2)
plt.plot(train_fractions * 100, val_scores,
         label="Validation Accuracy", color="orange",  marker="s", linewidth=2)

plt.title("Learning Curve: Gaussian Naive Bayes (HOG Features)", fontsize=14)
plt.xlabel("Percentage of Training Data Used (%)", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.ylim(0.5, 1.05)
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("learning_curve_naive_bayes.png", dpi=150)
plt.show()

# ─────────────────────────────────────────
# BIAS-VARIANCE DIAGNOSIS (printed)
# ─────────────────────────────────────────
final_train = train_scores[-1]
final_val   = val_scores[-1]
gap         = final_train - final_val

print(f"\n{'='*45}")
print(f"  BIAS-VARIANCE DIAGNOSIS")
print(f"{'='*45}")
print(f"  Final Train Accuracy : {final_train:.4f}")
print(f"  Final Val   Accuracy : {final_val:.4f}")
print(f"  Gap (overfit measure): {gap:.4f}")

if final_train < 0.85:
    print("\n  >> HIGH BIAS (Underfitting)")
    print("     Try: use a richer feature representation or relax var_smoothing")
elif gap > 0.08:
    print("\n  >> HIGH VARIANCE (Overfitting)")
    print("     Try: increase var_smoothing")
else:
    print("\n  >> GOOD FIT")
    print("     Train and val accuracy are close — well balanced")